In [1]:
import sys
sys.path.append("..")            # 저장소 루트 (project 패키지)
sys.path.append("../scripts")    # eval_silver 변환 함수 재사용

In [2]:
from pathlib import Path

BENCH_ID = "AIHub_CounselingSpeech_counsel_clean"
SILVER   = "/data/ASR/BENCHMARK/SILVER/AIHub_CounselingSpeech/transcript.jsonl"
MODEL    = "openai/whisper-small"
DEVICE   = "cuda:2"              # 실행 직전 nvidia-smi 로 빈 GPU 확인 후 지정

OUT_DIR = Path(f"../BENCHMARK/results/whisper_small__{BENCH_ID}")

In [3]:
from eval_silver import convert_silver

conv = OUT_DIR / "_silver_converted" / f"{BENCH_ID}.jsonl"
n = convert_silver(Path(SILVER), conv, corpus_id=BENCH_ID)
print(f"{n} samples → {conv}")

  [skip] 정답 전사가 빈 발화 2개 제외
19211 samples → ../BENCHMARK/results/whisper_small__AIHub_CounselingSpeech_counsel_clean/_silver_converted/AIHub_CounselingSpeech_counsel_clean.jsonl


In [4]:
from project.data.adapters.whisper import build_predict_fn

predict_fn = build_predict_fn(
    MODEL, backbone=MODEL,
    language="ko", task="transcribe",
    beam_size=5, batch_size=16, device=DEVICE,
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [5]:
from project.evaluation import evaluate_on_benchmark_suite

results = evaluate_on_benchmark_suite(
    model_name=f"whisper_small__{BENCH_ID}",
    predict_fn=predict_fn,
    benchmark_paths={BENCH_ID: conv},
    out_dir=OUT_DIR,
    batch_size=16,
)
results[BENCH_ID]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take p

CerResult(cer=19.741552360593886, scer=20.924144769459595, wer=48.53967572210888, samples=19211, per_sample_cer=[13.043478260869565, 20.833333333333336, 12.5, 5.555555555555555, 17.073170731707318, 10.465116279069768, 12.5, 9.230769230769232, 6.666666666666667, 15.384615384615385, 11.11111111111111, 16.666666666666664, 2.941176470588235, 6.25, 9.67741935483871, 6.329113924050633, 16.666666666666664, 12.121212121212121, 16.666666666666664, 6.0, 0.0, 9.375, 8.571428571428571, 16.666666666666664, 30.434782608695656, 13.333333333333334, 19.230769230769234, 2.857142857142857, 12.121212121212121, 13.043478260869565, 28.57142857142857, 5.263157894736842, 9.166666666666666, 3.8461538461538463, 33.33333333333333, 22.22222222222222, 7.4074074074074066, 20.0, 18.91891891891892, 6.25, 5.063291139240507, 8.21917808219178, 16.666666666666664, 20.588235294117645, 0.0, 16.666666666666664, 16.666666666666664, 16.0, 13.636363636363635, 16.0, 23.809523809523807, 8.695652173913043, 26.31578947368421, 171.

In [6]:
print((OUT_DIR / "evaluation_report.txt").read_text(encoding="utf-8"))

📊 ASR Evaluation Report — whisper_small__AIHub_CounselingSpeech_counsel_clean
   Date: 2026-06-16T14:41:08

## 1. Benchmark Set Results (한국어 CER 표준)
--------------------------------------------------------------------------------
Benchmark                                                  CER (%)   sCER (%)    Samples
--------------------------------------------------------------------------------
AIHub_CounselingSpeech_counsel_clean                         19.74      20.92     19,211
--------------------------------------------------------------------------------
Weighted Average                                             19.74                19,211

## 2. Slice Analysis (메타 필드별)
--------------------------------------------------------------------------------

### AIHub_CounselingSpeech_counsel_clean
  [by age_group]
  value                   CER (%)    samples
  10대                       27.66        706
  60대                       21.65      1,374
  20대                       20.69  

In [7]:
import json
import pandas as pd
import jiwer

lines = (OUT_DIR / BENCH_ID / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
df = pd.DataFrame(json.loads(l) for l in lines if l)

df["cer"] = [
    jiwer.cer(r, h) * 100 if r else float("nan")
    for r, h in zip(df["text_normalized"], df["prediction_normalized"])
]

worst = df.sort_values("cer", ascending=False).head(20)
for _, row in worst.iterrows():
    print(f"[CER {row.cer:5.1f}] 정답: {row.text_normalized}")
    print(f"             예측: {row.prediction_normalized}\n")

[CER 4975.0] 정답: 됐어요.
             예측: 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네

[CER 4958.3] 정답: 친절한 상담 감사해요.
             예측: 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다

[CER 4546.2] 정답: 네, 감사합니다. 결 결
             예측: 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니

In [8]:
import re, jiwer

def strip_spaces_in_numbers(t):
    # "일 조 이천 구백" 처럼 띄어쓴 한글 숫자를 비교에서 덜 불리하게:
    # 간단버전 — 숫자 인접 공백 제거로 표기차 일부 흡수
    return re.sub(r"(?<=\d)\s+(?=\d)", "", t)

# 더 정확히는 한글 수사 ↔ 아라비아 변환이 필요하지만,
# 우선 숫자가 포함된 발화와 아닌 발화의 CER을 분리해서 영향도부터 확인:
has_num = df["text_normalized"].str.contains(r"[0-9]|영|일|이|삼|사|오|육|칠|팔|구|십|백|천|만|조")
print(f"숫자 포함 발화: {has_num.sum()}개, CER {df.loc[has_num,'cer'].mean():.1f}%")
print(f"숫자 없는 발화: {(~has_num).sum()}개, CER {df.loc[~has_num,'cer'].mean():.1f}%")

숫자 포함 발화: 14018개, CER 22.7%
숫자 없는 발화: 5193개, CER 28.7%
